# Session 3 — OpenCV Basics (Part 1)
**Phase 1 · Foundations | Week 2 · Monday**

---

## What We'll Cover Today

| # | Topic | Key Concept |
|---|-------|-------------|
| 1 | Setup & Imports | Loading OpenCV in Python |
| 2 | `cv2.imread` | Reading images from disk |
| 3 | Image Properties | Shape, dtype, pixel access |
| 4 | `cv2.imshow` & `cv2.imwrite` | Displaying and saving images |
| 5 | `cv2.resize` | Changing image dimensions |
| 6 | Cropping | Slicing a region with NumPy |
| 7 | ROI (Region of Interest) | Extracting and working with sub-regions |
| 8 | Drawing Shapes & Text | Lines, rectangles, circles, text |
| 9 | Color Space Conversion | BGR → Gray / HSV (recap from Session 2) |
| 10 | Mini Exercise | Putting it all together |

---

> **Reminder from Session 2:**  
> OpenCV loads images in **BGR** order (not RGB).  
> Every image is a **NumPy array** of shape `(height, width, channels)`.

---
## 1. Setup & Imports

In [ ]:
# ── Install OpenCV if not already installed ──────────────────────────────────
# Uncomment and run once if needed:
# !pip install opencv-python matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt

print(f"OpenCV version : {cv2.__version__}")
print(f"NumPy  version : {np.__version__}")
print("All imports successful!")

In [ ]:
# ── Helper: display images inline in Jupyter ─────────────────────────────────
# OpenCV uses BGR; Matplotlib expects RGB, so we always convert before plotting.

def show(img, title="Image", cmap=None):
    """Display a single OpenCV image (BGR or grayscale) using Matplotlib."""
    plt.figure(figsize=(6, 5))
    if len(img.shape) == 3:
        # Colour image: convert BGR → RGB for Matplotlib
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        # Grayscale image
        plt.imshow(img, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def show_multiple(images, titles, cols=3, figsize=(14, 4)):
    """Display multiple images side by side."""
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 3:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            axes[i].imshow(img, cmap="gray")
        axes[i].set_title(title, fontsize=10)
        axes[i].axis("off")
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()

print("Helper functions defined.")

---
## 2. Creating a Test Image

Since we don't always have a specific image file handy, we'll **create synthetic test images** using NumPy.  
We'll also show how to download a sample image from the web.

You can replace these with your own images — just update the file paths.

In [ ]:
# ── Option A: Create a synthetic colour image with NumPy ─────────────────────
# We build a 300×400 RGB canvas and paint four coloured quadrants.
# Note: OpenCV uses BGR, so we specify (B, G, R) tuples.

h, w = 300, 400
synthetic = np.zeros((h, w, 3), dtype=np.uint8)

# Top-left  → Red    (BGR: 0, 0, 255)
synthetic[0 : h//2, 0 : w//2]   = (0,   0,   255)
# Top-right → Green  (BGR: 0, 255, 0)
synthetic[0 : h//2, w//2 : w]   = (0,   255, 0  )
# Bot-left  → Blue   (BGR: 255, 0, 0)
synthetic[h//2 : h, 0 : w//2]   = (255, 0,   0  )
# Bot-right → Yellow (BGR: 0, 255, 255)
synthetic[h//2 : h, w//2 : w]   = (0,   255, 255)

# Save it so we can practice imread later
cv2.imwrite("test_image.jpg", synthetic)
print("Saved test_image.jpg")
show(synthetic, "Synthetic test image (4 quadrants)")

In [ ]:
# ── Option B: Download a real photo (requires internet) ──────────────────────
# This uses urllib to grab a small sample image.

import urllib.request

url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg"
try:
    urllib.request.urlretrieve(url, "dog.jpg")
    print("Downloaded dog.jpg")
except Exception as e:
    print(f"Download failed ({e}). We will use test_image.jpg instead.")

---
## 3. `cv2.imread` — Reading Images from Disk

```python
cv2.imread(filename, flag)
```

| Flag | Value | What it does |
|------|-------|--------------|
| `cv2.IMREAD_COLOR` | 1 (default) | Load as colour (BGR, ignores alpha) |
| `cv2.IMREAD_GRAYSCALE` | 0 | Load as single-channel grayscale |
| `cv2.IMREAD_UNCHANGED` | -1 | Load as-is (keeps alpha if present) |

> **Important:** If the file path is wrong, `imread` returns `None` — it does **NOT** raise an error!

In [ ]:
# ── Read image in different modes ─────────────────────────────────────────────

img_color = cv2.imread("test_image.jpg", cv2.IMREAD_COLOR)       # default
img_gray  = cv2.imread("test_image.jpg", cv2.IMREAD_GRAYSCALE)   # grayscale
img_unch  = cv2.imread("test_image.jpg", cv2.IMREAD_UNCHANGED)   # unchanged

# Always check for None — common beginner mistake!
if img_color is None:
    raise FileNotFoundError("Image not found. Check the file path.")

print("img_color shape  :", img_color.shape)   # (H, W, 3)
print("img_gray  shape  :", img_gray.shape)    # (H, W)
print("img_unch  shape  :", img_unch.shape)    # (H, W, 3) for JPEG (no alpha)

show_multiple(
    [img_color, img_gray, img_unch],
    ["IMREAD_COLOR", "IMREAD_GRAYSCALE", "IMREAD_UNCHANGED"]
)

---
## 4. Image Properties & Pixel Access

Because images are NumPy arrays, all the properties we explored in Session 1 apply directly.

In [ ]:
# ── Image properties ──────────────────────────────────────────────────────────
img = cv2.imread("test_image.jpg")

height, width, channels = img.shape
print(f"Shape     : {img.shape}   → (height, width, channels)")
print(f"Height    : {height} px")
print(f"Width     : {width} px")
print(f"Channels  : {channels}")
print(f"Data type : {img.dtype}")
print(f"Total pixels : {img.size // channels:,}  ({height} × {width})")

In [ ]:
# ── Pixel access ──────────────────────────────────────────────────────────────
# Access a single pixel: img[row, col]  → returns [B, G, R]

pixel = img[50, 50]              # row=50, col=50
print(f"Pixel at (row=50, col=50): BGR = {pixel}")

# Access individual channels at a pixel
b = img[50, 50, 0]   # Blue channel
g = img[50, 50, 1]   # Green channel
r = img[50, 50, 2]   # Red channel
print(f"  B={b}, G={g}, R={r}")

# Modify a pixel (turn pixel white)
img_copy = img.copy()
img_copy[50, 50] = [255, 255, 255]   # set to white
print(f"After modification: {img_copy[50, 50]}")

# ── Extract a single channel across entire image ──────────────────────────────
blue_channel  = img[:, :, 0]   # All rows, all cols, channel 0 (Blue)
green_channel = img[:, :, 1]
red_channel   = img[:, :, 2]

show_multiple(
    [blue_channel, green_channel, red_channel],
    ["Blue Channel", "Green Channel", "Red Channel"]
)

---
## 5. `cv2.imshow` and `cv2.imwrite`

### `cv2.imshow` — Display in a pop-up window
```python
cv2.imshow("Window Title", image)
cv2.waitKey(0)       # 0 = wait forever;  N = wait N milliseconds
cv2.destroyAllWindows()
```

> In **Jupyter** we use `plt.imshow()` instead (the `show()` helper above).  
> `cv2.imshow()` is mainly used in standalone Python scripts.

### `cv2.imwrite` — Save to disk
```python
cv2.imwrite("output.jpg", image)   # returns True if successful
```

In [ ]:
# ── imwrite: save with different formats ─────────────────────────────────────

img = cv2.imread("test_image.jpg")

# Save as PNG (lossless)
success_png = cv2.imwrite("output.png", img)
print(f"Saved PNG  : {success_png}")

# Save as JPEG with quality parameter (0=worst → 100=best, default=95)
jpeg_params = [cv2.IMWRITE_JPEG_QUALITY, 80]
success_jpg = cv2.imwrite("output_q80.jpg", img, jpeg_params)
print(f"Saved JPEG (quality=80): {success_jpg}")

# Load both back and compare sizes visually
img_png = cv2.imread("output.png")
img_jpg = cv2.imread("output_q80.jpg")
show_multiple([img_png, img_jpg], ["PNG (lossless)", "JPEG quality=80"])

---
## 6. `cv2.resize` — Resizing Images

```python
cv2.resize(src, dsize, fx, fy, interpolation)
```

| Parameter | Description |
|-----------|-------------|
| `dsize` | Target `(width, height)` — note: width first! |
| `fx, fy` | Scale factors (use instead of dsize) |
| `interpolation` | Algorithm for resizing |

### Interpolation Methods

| Method | Use Case |
|--------|----------|
| `cv2.INTER_NEAREST` | Fastest, pixelated look (good for labels/masks) |
| `cv2.INTER_LINEAR` | Default, good for most cases |
| `cv2.INTER_CUBIC` | Slower, better quality for upscaling |
| `cv2.INTER_AREA` | Best for **downscaling** (avoids moire patterns) |
| `cv2.INTER_LANCZOS4` | Highest quality upscaling |

> **Tip:** Always use `INTER_AREA` when shrinking and `INTER_CUBIC` or `INTER_LANCZOS4` when enlarging.

In [ ]:
# ── Resize to specific dimensions ─────────────────────────────────────────────
img = cv2.imread("test_image.jpg")
print(f"Original shape: {img.shape}   (H x W x C)")

# Resize to fixed size — note dsize is (width, height), opposite of shape!
resized_small = cv2.resize(img, (200, 150), interpolation=cv2.INTER_AREA)
resized_large = cv2.resize(img, (800, 600), interpolation=cv2.INTER_CUBIC)

print(f"Small shape   : {resized_small.shape}")
print(f"Large shape   : {resized_large.shape}")

show_multiple(
    [img, resized_small, resized_large],
    [f"Original {img.shape[1]}x{img.shape[0]}",
     f"Small 200x150",
     f"Large 800x600"]
)

In [ ]:
# ── Resize by scale factor ────────────────────────────────────────────────────
img = cv2.imread("test_image.jpg")

# fx=0.5, fy=0.5 → half the size
half = cv2.resize(img, dsize=(0, 0), fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)

# fx=2.0, fy=2.0 → double the size
double = cv2.resize(img, dsize=(0, 0), fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)

print(f"Original : {img.shape}")
print(f"Half     : {half.shape}")
print(f"Double   : {double.shape}")

show_multiple([img, half, double],
              ["Original", "50% (fx=fy=0.5)", "200% (fx=fy=2.0)"])

In [ ]:
# ── Preserve aspect ratio when resizing to a target width ────────────────────
# Useful when you want to resize without distortion

def resize_to_width(img, target_width):
    """Resize image to a target width while preserving aspect ratio."""
    h, w = img.shape[:2]
    scale   = target_width / w
    target_height = int(h * scale)
    return cv2.resize(img, (target_width, target_height), interpolation=cv2.INTER_AREA)


img = cv2.imread("test_image.jpg")
resized = resize_to_width(img, target_width=250)

print(f"Original : {img.shape}")
print(f"Resized  : {resized.shape}")
show_multiple([img, resized], ["Original", "Resized to width=250 (AR preserved)"])

---
## 7. Cropping — Slicing a Region

Cropping is just **NumPy array slicing** — there's no dedicated OpenCV function for it.

```python
crop = img[y_start : y_end, x_start : x_end]
#           ─── rows ────    ─── columns ───
```

> **Remember:** Row index = Y direction (vertical), Column index = X direction (horizontal).  
> This is the **opposite** of screen coordinate convention (x, y).

In [ ]:
# ── Basic cropping ────────────────────────────────────────────────────────────
img = cv2.imread("test_image.jpg")

h, w = img.shape[:2]
print(f"Image size: {w}x{h}  (width x height)")

# Crop the top-left quadrant
crop_tl = img[0 : h//2, 0 : w//2]

# Crop the centre region
margin_y, margin_x = h // 4, w // 4
crop_center = img[margin_y : h - margin_y, margin_x : w - margin_x]

# Crop bottom-right quadrant
crop_br = img[h//2 : h, w//2 : w]

show_multiple(
    [img, crop_tl, crop_center, crop_br],
    ["Original", "Top-left quadrant", "Centre crop", "Bottom-right quadrant"],
    cols=4, figsize=(16, 4)
)

In [ ]:
# ── Important: crop is a VIEW, not a copy! ────────────────────────────────────
# If you modify the crop, you modify the original too.

img = cv2.imread("test_image.jpg")

crop_view = img[0:100, 0:100]          # This is a VIEW (shares memory)
crop_copy = img[0:100, 0:100].copy()   # This is an independent COPY

# Modify the view
crop_view[:] = 0   # Fill with black

print("After modifying crop_view:")
print(f"  img[0:5, 0:5] = {img[0:5, 0:5, 0]}  (original IS changed!)")

# crop_copy is unaffected
print(f"  crop_copy[0:3, 0:3] = {crop_copy[0:3, 0:3, 0]}  (copy is safe)")

show_multiple([img, crop_copy], ["Original (modified by view)", "Saved copy (unchanged)"])

---
## 8. ROI — Region of Interest

An **ROI** is a specific part of the image you want to focus on and process separately.  
It's one of the most fundamental concepts in Computer Vision pipelines.

Common use cases:
- Detecting a face → crop it → run a separate algorithm on just the face
- Reading a license plate → extract the plate region → run OCR
- Counting objects in a specific zone of a video frame

In [ ]:
# ── Extract and highlight an ROI ──────────────────────────────────────────────
img = cv2.imread("test_image.jpg")

# Define ROI as (x, y, w, h) — top-left corner + width/height
# This is the most common convention in OpenCV (used in detection bounding boxes)
roi_x, roi_y, roi_w, roi_h = 80, 50, 200, 150

# Extract the ROI
roi = img[roi_y : roi_y + roi_h, roi_x : roi_x + roi_w].copy()

# Draw a rectangle on the original to mark where the ROI is
img_marked = img.copy()
cv2.rectangle(img_marked,
              (roi_x, roi_y),                    # top-left corner
              (roi_x + roi_w, roi_y + roi_h),    # bottom-right corner
              (0, 255, 255),                      # colour BGR (yellow)
              thickness=3)

show_multiple([img_marked, roi],
              ["Image with ROI marked (yellow box)", "Extracted ROI"])

In [ ]:
# ── Copy an ROI to a different location ───────────────────────────────────────
# Classic OpenCV trick: paste one part of the image onto another.

img = cv2.imread("test_image.jpg")
h, w = img.shape[:2]

# Extract top-left quadrant
roi = img[0 : h//2, 0 : w//2].copy()

# Paste it into the bottom-right quadrant
dst = img.copy()
dst[h//2 : h, w//2 : w] = roi

show_multiple([img, dst],
              ["Original", "Top-left ROI pasted into bottom-right"])

In [ ]:
# ── Blur an ROI (practical privacy use case) ──────────────────────────────────
# Simulate blurring a "sensitive" region — like a face or license plate.

img = cv2.imread("test_image.jpg")

# Define the region to blur
bx, by, bw, bh = 100, 75, 180, 120

blurred = img.copy()
roi_region = blurred[by : by + bh, bx : bx + bw]

# Apply Gaussian blur to the ROI only
roi_blurred = cv2.GaussianBlur(roi_region, ksize=(31, 31), sigmaX=0)
blurred[by : by + bh, bx : bx + bw] = roi_blurred

show_multiple([img, blurred],
              ["Original", "ROI blurred (privacy effect)"])

---
## 9. Drawing Shapes & Text

All drawing functions work **in-place** on the image array. They return the modified array too, but they **also modify** the original. Always work on a `.copy()` to keep the original safe.

```python
cv2.line(img, pt1, pt2, color, thickness)
cv2.rectangle(img, pt1, pt2, color, thickness)   # -1 = filled
cv2.circle(img, center, radius, color, thickness)
cv2.ellipse(img, center, axes, angle, startAngle, endAngle, color, thickness)
cv2.polylines(img, pts, isClosed, color, thickness)
cv2.fillPoly(img, pts, color)
cv2.putText(img, text, org, fontFace, fontScale, color, thickness)
```

> All coordinates are **(x, y)** tuples (column first, row second).  
> All colours are **(B, G, R)** tuples.

In [ ]:
# ── Create a blank white canvas for drawing ───────────────────────────────────
canvas = np.ones((500, 700, 3), dtype=np.uint8) * 240   # light grey background

# ── 1. Line ───────────────────────────────────────────────────────────────────
# cv2.line(img, pt1, pt2, color_BGR, thickness)
cv2.line(canvas,
         pt1=(30, 50),      # start point (x, y)
         pt2=(300, 50),     # end point (x, y)
         color=(0, 0, 200), # BGR red
         thickness=3)

# Dashed line — OpenCV has no native dashed line, so we draw segments manually
for x in range(30, 300, 20):
    cv2.line(canvas, (x, 80), (x + 10, 80), (0, 150, 0), 2)

cv2.putText(canvas, "Lines", (30, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50, 50, 50), 2)

# ── 2. Rectangles ─────────────────────────────────────────────────────────────
# Hollow rectangle
cv2.rectangle(canvas,
              pt1=(30, 110),
              pt2=(200, 200),
              color=(200, 0, 0),   # BGR blue
              thickness=3)

# Filled rectangle (thickness=-1)
cv2.rectangle(canvas,
              pt1=(220, 110),
              pt2=(390, 200),
              color=(0, 200, 200), # BGR yellow
              thickness=-1)        # -1 = filled

cv2.putText(canvas, "Rectangles", (30, 105),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50, 50, 50), 2)

# ── 3. Circles ────────────────────────────────────────────────────────────────
# Hollow circle
cv2.circle(canvas,
           center=(100, 300),
           radius=60,
           color=(0, 0, 180),
           thickness=3)

# Filled circle
cv2.circle(canvas,
           center=(250, 300),
           radius=50,
           color=(180, 100, 0),   # dark cyan
           thickness=-1)

cv2.putText(canvas, "Circles", (30, 220),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50, 50, 50), 2)

# ── 4. Ellipse ────────────────────────────────────────────────────────────────
# cv2.ellipse(img, center, axes, angle, startAngle, endAngle, color, thickness)
cv2.ellipse(canvas,
            center=(420, 300),
            axes=(100, 50),       # (semi-major, semi-minor) radii
            angle=30,             # rotation of the ellipse
            startAngle=0,
            endAngle=360,
            color=(150, 0, 150),
            thickness=3)

# Draw only an arc (partial ellipse)
cv2.ellipse(canvas,
            center=(600, 300),
            axes=(60, 40),
            angle=0,
            startAngle=0,
            endAngle=270,         # only 270 degrees
            color=(0, 180, 80),
            thickness=3)

cv2.putText(canvas, "Ellipses", (390, 220),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50, 50, 50), 2)

# ── 5. Polygon ────────────────────────────────────────────────────────────────
# Points must be a numpy array of shape (N, 1, 2) and dtype int32
triangle_pts = np.array([[500, 120], [620, 200], [450, 200]], dtype=np.int32)
triangle_pts = triangle_pts.reshape((-1, 1, 2))   # required shape

cv2.polylines(canvas,
              [triangle_pts],
              isClosed=True,
              color=(0, 100, 200),
              thickness=3)

# Filled polygon
star_pts = np.array(
    [[560, 110], [575, 155], [620, 155], [585, 180],
     [600, 225], [560, 200], [520, 225], [535, 180],
     [500, 155], [545, 155]], dtype=np.int32
)
cv2.fillPoly(canvas, [star_pts.reshape(-1, 1, 2)], color=(0, 200, 255))

cv2.putText(canvas, "Polygons", (450, 105),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50, 50, 50), 2)

# ── 6. Text ───────────────────────────────────────────────────────────────────
cv2.putText(canvas,
            text="cv2.putText",
            org=(30, 440),              # bottom-left of text box
            fontFace=cv2.FONT_HERSHEY_SIMPLEX,
            fontScale=1.2,
            color=(0, 50, 200),
            thickness=2,
            lineType=cv2.LINE_AA)       # anti-aliased edges

# Other font faces:
cv2.putText(canvas, "FONT_HERSHEY_COMPLEX",
            (280, 440), cv2.FONT_HERSHEY_COMPLEX, 0.8, (100, 0, 100), 2)

show(canvas, "OpenCV Drawing Functions")

In [ ]:
# ── Annotating a real image ───────────────────────────────────────────────────
# This is what it looks like in a real CV pipeline (bounding box + label)

img = cv2.imread("test_image.jpg")
annotated = img.copy()

# Simulate a "detection" bounding box
bbox = (50, 40, 180, 130)   # (x, y, w, h)
x, y, bw, bh = bbox
label = "Object (0.94)"

# Draw filled background for the label
(text_w, text_h), baseline = cv2.getTextSize(
    label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2
)
cv2.rectangle(annotated, (x, y - text_h - baseline - 4), (x + text_w, y),
              (0, 180, 0), -1)   # green filled rectangle for label background

# Draw bounding box
cv2.rectangle(annotated, (x, y), (x + bw, y + bh), (0, 180, 0), 2)

# Draw label text
cv2.putText(annotated, label, (x, y - 5),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)

# Draw a centre dot
cx, cy = x + bw // 2, y + bh // 2
cv2.circle(annotated, (cx, cy), 5, (0, 0, 255), -1)

show_multiple([img, annotated],
              ["Original", "Annotated (detection-style)"])

---
## 10. Colour Space Conversion (Quick Recap from Session 2)

Now that we know how to load and display images, let's revisit colour spaces with real code.

```python
cv2.cvtColor(src, code)
```

In [ ]:
# ── Colour space conversions ──────────────────────────────────────────────────
img = cv2.imread("test_image.jpg")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
lab  = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

print(f"BGR  shape: {img.shape}    dtype: {img.dtype}")
print(f"Gray shape: {gray.shape}   dtype: {gray.dtype}")
print(f"HSV  shape: {hsv.shape}    dtype: {hsv.dtype}")
print(f"LAB  shape: {lab.shape}    dtype: {lab.dtype}")

# Show each conversion
# Note: HSV and LAB won't look meaningful as RGB, but we show them for completeness
show_multiple(
    [img, gray, hsv, lab],
    ["BGR (original)", "Grayscale", "HSV (raw bytes)", "LAB (raw bytes)"],
    cols=4, figsize=(16, 3)
)

In [ ]:
# ── Practical: isolate a colour using HSV masking ─────────────────────────────
# This is a preview of what we'll do in depth in a later session.
# Goal: isolate only the GREEN region of our test image.

img = cv2.imread("test_image.jpg")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# Green hue range in OpenCV HSV:  H ≈ 35–85 (out of 0–179)
lower_green = np.array([35,  80,  80])
upper_green = np.array([85, 255, 255])

# Create a binary mask: 255 where green, 0 elsewhere
mask = cv2.inRange(hsv, lower_green, upper_green)

# Apply mask to original image
result = cv2.bitwise_and(img, img, mask=mask)

show_multiple(
    [img, mask, result],
    ["Original", "Green mask (binary)", "Green regions only"]
)

---
## 11. Bonus: `cv2.flip` and `cv2.rotate`

Simple geometric operations that come up constantly in data augmentation and image correction.

In [ ]:
# ── Flip ──────────────────────────────────────────────────────────────────────
# flipCode:  0 = vertical,  1 = horizontal,  -1 = both

img = cv2.imread("test_image.jpg")

flip_h  = cv2.flip(img,  1)   # Horizontal (mirror)
flip_v  = cv2.flip(img,  0)   # Vertical (upside down)
flip_hv = cv2.flip(img, -1)   # Both axes

show_multiple(
    [img, flip_h, flip_v, flip_hv],
    ["Original", "Flip H (flipCode=1)", "Flip V (flipCode=0)", "Both (flipCode=-1)"],
    cols=4, figsize=(16, 3)
)

In [ ]:
# ── Rotate ────────────────────────────────────────────────────────────────────
# cv2.rotate uses fixed 90-degree multiples

img = cv2.imread("test_image.jpg")

rot_90  = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
rot_180 = cv2.rotate(img, cv2.ROTATE_180)
rot_270 = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)

show_multiple(
    [img, rot_90, rot_180, rot_270],
    ["Original", "90° CW", "180°", "270° CW (=90° CCW)"],
    cols=4, figsize=(16, 4)
)

In [ ]:
# ── Rotate by arbitrary angle using warpAffine ────────────────────────────────
# For any angle (not just 90°), we use a rotation matrix.

def rotate_image(img, angle_degrees, expand=True):
    """
    Rotate image by an arbitrary angle around its centre.
    If expand=True, the output canvas grows to fit the rotated image.
    """
    h, w = img.shape[:2]
    cx, cy = w // 2, h // 2   # rotation centre

    # Rotation matrix (2x3)
    M = cv2.getRotationMatrix2D(center=(cx, cy),
                                 angle=angle_degrees,
                                 scale=1.0)

    if expand:
        # Compute new canvas size to fit the rotated image
        cos_a = abs(M[0, 0])
        sin_a = abs(M[0, 1])
        new_w = int(h * sin_a + w * cos_a)
        new_h = int(h * cos_a + w * sin_a)
        M[0, 2] += (new_w / 2) - cx
        M[1, 2] += (new_h / 2) - cy
    else:
        new_w, new_h = w, h

    rotated = cv2.warpAffine(img, M, (new_w, new_h),
                             borderMode=cv2.BORDER_CONSTANT,
                             borderValue=(200, 200, 200))  # grey border
    return rotated


img = cv2.imread("test_image.jpg")

rot_30  = rotate_image(img,  30)
rot_45  = rotate_image(img,  45)
rot_neg = rotate_image(img, -20)

show_multiple(
    [img, rot_30, rot_45, rot_neg],
    ["Original", "Rotated 30°", "Rotated 45°", "Rotated -20°"],
    cols=4, figsize=(16, 4)
)

---
## 12. Bonus: Image Arithmetic — Brightness & Contrast

Two essential operations you'll use constantly: adjusting **brightness** and **contrast**.

$$\text{output}(x,y) = \alpha \cdot \text{input}(x,y) + \beta$$

- $\alpha$ → controls **contrast** (1.0 = no change, >1 increases, <1 decreases)
- $\beta$ → controls **brightness** (0 = no change, positive = brighter)

In [ ]:
# ── Brightness and Contrast adjustment ───────────────────────────────────────
img = cv2.imread("test_image.jpg")

# convertScaleAbs handles clipping automatically (keeps values in 0–255)
brighter      = cv2.convertScaleAbs(img, alpha=1.0, beta=60)   # +60 brightness
darker        = cv2.convertScaleAbs(img, alpha=1.0, beta=-60)  # -60 brightness
high_contrast = cv2.convertScaleAbs(img, alpha=1.5, beta=0)    # 1.5x contrast
low_contrast  = cv2.convertScaleAbs(img, alpha=0.5, beta=60)   # dimmer + lift

show_multiple(
    [img, brighter, darker, high_contrast, low_contrast],
    ["Original", "Brighter (beta=+60)", "Darker (beta=-60)",
     "High contrast (alpha=1.5)", "Low contrast (alpha=0.5)"],
    cols=5, figsize=(18, 3)
)

---
## 13. Mini Exercise — Putting It All Together

Apply everything you've learnt today in one small pipeline.

**Task:** Given `test_image.jpg`:
1. Load the image
2. Resize it to 320×240
3. Extract the top-left quadrant as an ROI
4. Draw a green circle at the centre of the ROI
5. Paste the annotated ROI back into the bottom-right corner of the full image
6. Add a white text label "Session 3" at the top-left
7. Save the result as `exercise_output.jpg` and display it

In [ ]:
# ── Mini Exercise Solution ────────────────────────────────────────────────────
# Try to complete this yourself first. Then check the solution below.

# Step 1: Load
img = cv2.imread("test_image.jpg")

# Step 2: Resize to 320×240
img = cv2.resize(img, (320, 240), interpolation=cv2.INTER_AREA)
h, w = img.shape[:2]
print(f"Resized to: {w}x{h}")

# Step 3: Extract top-left quadrant ROI
roi = img[0 : h//2, 0 : w//2].copy()

# Step 4: Draw green circle at centre of ROI
roi_h, roi_w = roi.shape[:2]
cx, cy = roi_w // 2, roi_h // 2
cv2.circle(roi, center=(cx, cy), radius=30,
           color=(0, 255, 0), thickness=3)
cv2.circle(roi, center=(cx, cy), radius=5,
           color=(0, 255, 0), thickness=-1)   # filled dot at exact centre

# Step 5: Paste annotated ROI into bottom-right corner
result = img.copy()
result[h - roi_h : h, w - roi_w : w] = roi

# Draw a border around the pasted region so it's visible
cv2.rectangle(result,
              (w - roi_w, h - roi_h), (w - 1, h - 1),
              color=(0, 255, 255), thickness=2)

# Step 6: Add text label
cv2.putText(result,
            text="Session 3",
            org=(10, 25),
            fontFace=cv2.FONT_HERSHEY_SIMPLEX,
            fontScale=0.8,
            color=(255, 255, 255),
            thickness=2,
            lineType=cv2.LINE_AA)

# Step 7: Save and display
cv2.imwrite("exercise_output.jpg", result)
print("Saved exercise_output.jpg")
show(result, "Mini Exercise Output")

---
## 14. Quick Reference Card

```python
import cv2
import numpy as np

# ── Read / Write / Display ────────────────────────────────────────────────────
img = cv2.imread("file.jpg")             # Read (BGR, returns None if not found)
img = cv2.imread("file.jpg", 0)          # Grayscale
cv2.imwrite("out.jpg", img)              # Save
cv2.imshow("title", img)                 # Pop-up window (scripts only)
cv2.waitKey(0); cv2.destroyAllWindows()  # Essential after imshow

# ── Properties ────────────────────────────────────────────────────────────────
h, w, c = img.shape       # height, width, channels
pixel   = img[row, col]   # BGR values at (row, col)

# ── Resize ────────────────────────────────────────────────────────────────────
cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)   # shrink
cv2.resize(img, (0,0), fx=0.5, fy=0.5)                          # by scale

# ── Crop & ROI ────────────────────────────────────────────────────────────────
crop = img[y1:y2, x1:x2].copy()   # always .copy() if you'll modify it

# ── Drawing ───────────────────────────────────────────────────────────────────
cv2.line(img, (x1,y1), (x2,y2), (B,G,R), thickness)
cv2.rectangle(img, (x1,y1), (x2,y2), (B,G,R), thickness)  # -1 = filled
cv2.circle(img, (cx,cy), radius, (B,G,R), thickness)
cv2.putText(img, "text", (x,y), cv2.FONT_HERSHEY_SIMPLEX, scale, (B,G,R), t)

# ── Colour space ─────────────────────────────────────────────────────────────
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # for Matplotlib

# ── Flip & Rotate ─────────────────────────────────────────────────────────────
cv2.flip(img, 1)                              # horizontal mirror
cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)      # 90° CW
```

---

## What's Next — Session 4: Filtering & Edge Detection

- **Blurring:** Gaussian, Median, Bilateral filters
- **Edge Detection:** Sobel, Laplacian, Canny
- **Thresholding:** Simple, Otsu, Adaptive
- **Morphological Operations:** Erosion, Dilation, Opening, Closing